# SpotLight baseline on TelecomTS

Paper-aligned re-implementation of the **SpotLight** anomaly-detection
pipeline (Sun et al., *MobiCom* '24, https://doi.org/10.1145/3636534.3649380)
used as a baseline against KAC.

**Pipeline (paper Figure 3):**
- **Detection:** `JVGAN -> MRPI`
- **Explainability:** `KFilter -> CausalNex`

This notebook now treats `JVGAN + MRPI` as the primary anomaly detector and
keeps `KFilter` only for KPI persistence filtering / root-cause analysis.

**TelecomTS setup:**
- Test split is the same `494`-window file used by the other baselines.
- SpotLight is trained on **normal-only** windows from train + val.
- Validation remains labelled so we can compare the raw MRPI detector, a
  validation-tuned MRPI cutoff, and the stricter KFilter rule.
- Checkpoints are versioned by dataset and run profile so stale pre-fix weights
  are not silently reused.

**Runtime strategy:**
- This notebook exposes `diagnostic`, `medium`, and `faithful` run profiles.
- Prediction runs in chunks with cache cleanup between chunks to keep a MacBook
  responsive.
- Causal explainability is optional and disabled during metric debugging because
  it does not affect detection metrics and is not supported in the default
  Python 3.12 macOS setup.

Input shape: `(N, 128, 16)` (16 KPIs × 128 timesteps, 247/247 normal/anom).


## 1. CausalNex note (optional)

In [ ]:
import importlib.util
import platform
import sys

causalnex_available = importlib.util.find_spec("causalnex") is not None
print(f"Python: {sys.version.split()[0]} | platform: {platform.platform()}")
if causalnex_available:
    print("causalnex already available; full explainability stage can be enabled.")
else:
    print("causalnex is unavailable in this environment; detection metrics will run without it.")
    print("If you need the causal stage later, use a Python 3.10/3.11 environment where causalnex is supported.")


## 2. Imports & device

In [ ]:
import sys, os, json, math, gc, time, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)

# Make the local spotlight_baseline package importable regardless of where
# Jupyter starts the kernel.
REPO_ROOT = Path("..") / ".."
sys.path.insert(0, str(REPO_ROOT.resolve()))

from spotlight_baseline import SpotLightPipeline, preprocess_windows  # noqa: E402

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "telecomts"
RUN_PROFILE = os.environ.get("SPOTLIGHT_RUN_PROFILE", "medium").strip().lower()
CKPT_SCHEMA_VERSION = "recovery_v2"
PROFILES = {
    "diagnostic": {
        "jvgan_epochs": 10,
        "mrpi_epochs": 5,
        "n_jvgan_samples": 16,
        "n_csdi_samples": 8,
        "jvgan_sample_batch_size": 4,
        "mrpi_sample_batch_size": 1,
        "predict_chunk_size": 32,
        "run_causal": False,
    },
    "medium": {
        "jvgan_epochs": 40,
        "mrpi_epochs": 20,
        "n_jvgan_samples": 64,
        "n_csdi_samples": 24,
        "jvgan_sample_batch_size": 4,
        "mrpi_sample_batch_size": 1,
        "predict_chunk_size": 16,
        "run_causal": False,
    },
    "faithful": {
        "jvgan_epochs": 100,
        "mrpi_epochs": 100,
        "n_jvgan_samples": 150,
        "n_csdi_samples": 100,
        "jvgan_sample_batch_size": 4,
        "mrpi_sample_batch_size": 1,
        "predict_chunk_size": 8,
        "run_causal": False,
    },
}
if RUN_PROFILE not in PROFILES:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose from {sorted(PROFILES)}")
CFG = PROFILES[RUN_PROFILE]
ckpt_token = hashlib.sha1(f"{DATASET_NAME}|{RUN_PROFILE}|{CKPT_SCHEMA_VERSION}".encode()).hexdigest()[:8]
CKPT_DIR = Path(f"ckpt_spotlight_{DATASET_NAME}_{RUN_PROFILE}_{ckpt_token}")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE} | torch {torch.__version__}")
print(f"Run profile: {RUN_PROFILE} -> {CFG}")
print(f"Checkpoint schema: {CKPT_SCHEMA_VERSION}")
print(f"Checkpoints will be written to: {CKPT_DIR.resolve()}")


## 3. SSL fix for HuggingFace (macOS) -- harmless on other platforms

In [ ]:
import ssl, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl._create_unverified_context
print("SSL cert bundle:", certifi.where())


## 4. Load dataset (same `*_test.npz` used by every other baseline)

In [ ]:
DATA_DIR = Path("data")
train_d = np.load(DATA_DIR / "TelecomTS_train.npz", allow_pickle=True)
val_d   = np.load(DATA_DIR / "TelecomTS_val.npz",   allow_pickle=True)
test_d  = np.load(DATA_DIR / "TelecomTS_test.npz",  allow_pickle=True)

# SpotLight is trained on normal-only data (paper sec. 3.2). We concatenate
# train+val *only for that*, then keep the labelled validation split separate
# so we can tune the post-MRPI window-score threshold on it (sec. 3.2.5: the
# paper itself empirically picks the 25% KFilter cutoff on its own dataset --
# we do the analogous tuning of the window-score cutoff here).
X_train_full = np.concatenate([train_d["X"], val_d["X"]], axis=0).astype(np.float32)
y_train_full = np.concatenate([train_d["y"], val_d["y"]]).astype(np.int64)
X_val  = val_d["X"].astype(np.float32)
y_val  = val_d["y"].astype(np.int64)
X_test = test_d["X"].astype(np.float32)
y_test = test_d["y"].astype(np.int64)

# SpotLight is trained on normal-only data.
X_train_normal = X_train_full[y_train_full == 0]
KPI_NAMES = list(map(str, train_d["feature_cols"])) if "feature_cols" in train_d.files else None

print(f"Normal train+val: {X_train_normal.shape}")
print(f"Val (labelled):   {X_val.shape}  (anom={int(y_val.sum())}, normal={int((y_val==0).sum())})")
print(f"Test:             {X_test.shape}  (anom={int(y_test.sum())}, normal={int((y_test==0).sum())})")


## 5. Preprocess (paper §3.2.3)
Nearest-neighbour missing-data imputation, per-KPI 0–1 normalization with training-set min/max, time alignment.

In [ ]:
# Paper §3.2.3: NN missing-data imputation -> per-KPI [0, 1] -> aligned.
X_train_norm, scaler = preprocess_windows(X_train_normal)
X_val_norm,   _      = preprocess_windows(X_val,         scaler=scaler)
X_test_norm,  _      = preprocess_windows(X_test,        scaler=scaler)

print("Preprocessed:")
print(f"  X_train_norm: {X_train_norm.shape}  range=[{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]")
print(f"  X_val_norm  : {X_val_norm.shape}   range=[{X_val_norm.min():.3f}, {X_val_norm.max():.3f}]")
print(f"  X_test_norm : {X_test_norm.shape}  range=[{X_test_norm.min():.3f}, {X_test_norm.max():.3f}]")

N_KPIS  = X_train_norm.shape[2]
SEQ_LEN = X_train_norm.shape[1]
print(f"  N_KPIS = {N_KPIS}, SEQ_LEN = {SEQ_LEN}")


## 6. Build pipeline
JVGAN + 11 CSDI models + KFilter + CausalNex. The run profile controls the training budget and inference sample counts so you can switch between `diagnostic`, `medium`, and `faithful` runs without rewriting the notebook.

In [ ]:
JVGAN_EPOCHS = CFG["jvgan_epochs"]
MRPI_EPOCHS  = CFG["mrpi_epochs"]
N_JVGAN_SAMPLES = CFG["n_jvgan_samples"]
N_CSDI_SAMPLES = CFG["n_csdi_samples"]
JVGAN_SAMPLE_BATCH_SIZE = CFG["jvgan_sample_batch_size"]
MRPI_SAMPLE_BATCH_SIZE = CFG["mrpi_sample_batch_size"]
PREDICT_CHUNK_SIZE = CFG["predict_chunk_size"]
RUN_CAUSAL = bool(CFG["run_causal"] and causalnex_available)

pipe = SpotLightPipeline(
    n_kpis=N_KPIS,
    seq_len=SEQ_LEN,
    device=DEVICE,
    jvgan_latent_continuous=16,
    jvgan_latent_discrete=(10, 10, 10),
    jvgan_dropout=0.2,
    mrpi_diffusion_steps=50,
    kfilter_min_fraction=0.25,
    causal_max_kpis=30,
    causal_lag_p=1,
)
print(
    f"Pipeline built. JVGAN_EPOCHS={JVGAN_EPOCHS}, MRPI_EPOCHS={MRPI_EPOCHS}, "
    f"JVGAN samples={N_JVGAN_SAMPLES}, MRPI samples={N_CSDI_SAMPLES}, "
    f"chunk_size={PREDICT_CHUNK_SIZE}, RUN_CAUSAL={RUN_CAUSAL}."
)


## 7. Train pipeline
JVGAN first, then the 11 CSDI imputation models in sequence. Checkpoints to `ckpt_spotlight/` -- already-trained components are skipped on re-run.

In [ ]:
# Train. JVGAN saves to {CKPT_DIR}/jvgan.pt; the 11 CSDI models save to
# {CKPT_DIR}/mrpi/csdi_interval_NN.pt. Already-trained components are skipped
# on re-run, so you can interrupt and resume.
history = pipe.fit(
    X_train_norm,
    jvgan_epochs=JVGAN_EPOCHS,
    jvgan_batch_size=64,
    jvgan_lr=1e-3,
    jvgan_adv_weight=0.1,
    mrpi_epochs=MRPI_EPOCHS,
    mrpi_batch_size=32,
    mrpi_lr=1e-3,
    checkpoint_dir=str(CKPT_DIR),
    verbose=True,
)
print(f"Training complete (or skipped due to checkpoints) for profile={RUN_PROFILE!r}.")


## 8. Predict on validation and test
The primary detector is `JVGAN + MRPI`. Prediction now runs in chunks with explicit cache cleanup between chunks so the notebook stays responsive while still exposing the stricter KFilter outputs for diagnostics.

In [ ]:
def _flush_device_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def _concat_predict_outputs(parts):
    array_keys = [
        "window_is_anomaly",
        "window_score",
        "jvgan_flagged",
        "confirmed",
        "kpi_anom_fraction",
        "mrpi_any_point_flag",
        "kfilter_window_is_anomaly",
        "kfilter_window_score",
    ]
    list_keys = ["kfilter_kpis", "root_cause_kpis"]
    out = {}
    for key in array_keys:
        out[key] = np.concatenate([part[key] for part in parts], axis=0)
    for key in list_keys:
        merged = []
        for part in parts:
            merged.extend(part[key])
        out[key] = merged
    return out


def predict_in_chunks(X, *, label, run_causal, verbose):
    n = len(X)
    chunk = min(PREDICT_CHUNK_SIZE, n)
    parts = []
    n_chunks = math.ceil(n / chunk)
    print(f"[{label}] running {n} windows in {n_chunks} chunk(s) of up to {chunk} ...")
    t0 = time.time()
    for idx, start in enumerate(range(0, n, chunk), start=1):
        end = min(start + chunk, n)
        print(f"[{label}] chunk {idx}/{n_chunks}: windows {start}:{end}")
        part = pipe.predict(
            X[start:end],
            n_jvgan_samples=N_JVGAN_SAMPLES,
            jvgan_sample_batch_size=JVGAN_SAMPLE_BATCH_SIZE,
            kappa=1.0,
            n_csdi_samples=N_CSDI_SAMPLES,
            mrpi_sample_batch_size=MRPI_SAMPLE_BATCH_SIZE,
            q_low=0.05,
            q_high=0.95,
            kpi_names=KPI_NAMES,
            run_causal=run_causal,
            verbose=verbose and idx == 1,
        )
        parts.append(part)
        _flush_device_cache()
        elapsed = time.time() - t0
        avg = elapsed / idx
        eta = avg * (n_chunks - idx)
        print(f"[{label}] finished chunk {idx}/{n_chunks} | elapsed={elapsed/60:.1f} min | eta~{eta/60:.1f} min")
    return _concat_predict_outputs(parts)


out = predict_in_chunks(X_test_norm, label="test", run_causal=RUN_CAUSAL, verbose=True)

window_is_anomaly = out["window_is_anomaly"]
window_score = out["window_score"]
kfilter_window_is_anomaly = out["kfilter_window_is_anomaly"]
kfilter_window_score = out["kfilter_window_score"]

print(
    f"Test (paper-faithful MRPI any-point rule): {int(window_is_anomaly.sum())} / {len(window_is_anomaly)} flagged anomalous"
)
print(f"Test score range: [{window_score.min():.3f}, {window_score.max():.3f}]")
print(
    f"Test (diagnostic KFilter rule >= {pipe.kfilter.min_fraction}): {int(kfilter_window_is_anomaly.sum())} / {len(kfilter_window_is_anomaly)} flagged anomalous"
)
print(f"Diagnostic KFilter score range: [{kfilter_window_score.min():.3f}, {kfilter_window_score.max():.3f}]")

print("\n[Threshold tuning] running pipeline on validation split ...")
val_out = predict_in_chunks(X_val_norm, label="val", run_causal=False, verbose=False)
val_score = val_out["window_score"]
val_kfilter_score = val_out["kfilter_window_score"]
print(f"Val score range: [{val_score.min():.3f}, {val_score.max():.3f}]")
print(f"Val KFilter score range: [{val_kfilter_score.min():.3f}, {val_kfilter_score.max():.3f}]")


## 9. Compute metrics & save

We report **three** rows:

1. **Paper-faithful detector**: `JVGAN + MRPI`, with `window_is_anomaly = any confirmed point`.
2. **Validation-tuned MRPI score cutoff**: tune the `window_score` cutoff on the labelled validation split.
3. **Diagnostic KFilter rule**: old stricter persistence-based window rule, kept only to show how much it differs from the corrected detector.

For threshold-free metrics, AUROC and AP are computed from the MRPI-based `window_score`.

In [ ]:
def best_f1_threshold(scores: np.ndarray, labels: np.ndarray, n_grid: int = 201,
                      min_pp: float = 0.02, max_pp: float = 0.98):
    """Pick the threshold that maximizes F1, skipping degenerate (all-pos / all-neg) cuts."""
    s = np.asarray(scores, dtype=np.float64)
    y = np.asarray(labels, dtype=np.int64)
    if s.size == 0:
        return 0.5, 0.0
    lo, hi = float(s.min()), float(s.max())
    if lo == hi:
        return lo, float(f1_score(y, np.zeros_like(y), zero_division=0))
    grid = np.linspace(lo, hi, n_grid)
    best_t, best_f1 = None, -1.0
    for t in grid:
        pred = (s >= t).astype(np.int64)
        frac = pred.mean()
        if frac < min_pp or frac > max_pp:
            continue
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = float(f1), float(t)
    if best_t is None:
        med = float(np.median(s))
        best_f1 = float(f1_score(y, (s >= med).astype(np.int64), zero_division=0))
        return med, best_f1
    return best_t, best_f1


def metric_row(method: str, labels: np.ndarray, preds: np.ndarray, scores: np.ndarray):
    return {
        "Method": method,
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
        "Accuracy": accuracy_score(labels, preds),
        "AUROC": roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else float("nan"),
        "AP": average_precision_score(labels, scores) if len(np.unique(labels)) > 1 else float("nan"),
    }


preds_paper = window_is_anomaly.astype(int)
m_paper = metric_row("SpotLight (paper-faithful MRPI any-point)", y_test, preds_paper, window_score)

tuned_thr, tuned_val_f1 = best_f1_threshold(val_score, y_val)
print(f"[Threshold tuning] best-F1 cutoff on val = {tuned_thr:.4f} (val F1 = {tuned_val_f1:.4f})")
preds_tuned = (window_score >= tuned_thr).astype(int)
m_tuned = metric_row(f"SpotLight (val-tuned MRPI cutoff = {tuned_thr:.3f})", y_test, preds_tuned, window_score)

preds_kfilter = kfilter_window_is_anomaly.astype(int)
m_kfilter = metric_row(
    f"SpotLight (diagnostic KFilter >= {pipe.kfilter.min_fraction})",
    y_test,
    preds_kfilter,
    kfilter_window_score,
)

print("\n=== Test metrics ===")
for label, m in [
    ("Paper-faithful detector", m_paper),
    ("Val-tuned MRPI cutoff", m_tuned),
    ("Diagnostic KFilter rule", m_kfilter),
]:
    print(f"\n[{label}]  {m['Method']}")
    for k in ("Precision", "Recall", "F1", "Accuracy", "AUROC", "AP"):
        print(f"  {k:10s} = {m[k]:.4f}")

metrics = m_tuned

df = pd.DataFrame([m_paper, m_tuned, m_kfilter])
csv_path = RESULTS_DIR / "spotlight_baseline_comparison.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved -> {csv_path}")

artifacts = {
    "run_profile": RUN_PROFILE,
    "device": DEVICE,
    "y_test": y_test.tolist(),
    "window_is_anomaly": window_is_anomaly.astype(int).tolist(),
    "window_score": window_score.astype(float).tolist(),
    "kfilter_window_is_anomaly": kfilter_window_is_anomaly.astype(int).tolist(),
    "kfilter_window_score": kfilter_window_score.astype(float).tolist(),
    "preds_paper": preds_paper.tolist(),
    "preds_tuned": preds_tuned.tolist(),
    "preds_kfilter": preds_kfilter.tolist(),
    "tuned_threshold": float(tuned_thr),
    "kfilter_kpis": out["kfilter_kpis"],
    "root_cause_kpis": out["root_cause_kpis"],
    "kpi_names": list(KPI_NAMES) if KPI_NAMES is not None else None,
    "metrics_paper": {k: (v if isinstance(v, str) else float(v)) for k, v in m_paper.items()},
    "metrics_tuned": {k: (v if isinstance(v, str) else float(v)) for k, v in m_tuned.items()},
    "metrics_kfilter": {k: (v if isinstance(v, str) else float(v)) for k, v in m_kfilter.items()},
}
json_path = RESULTS_DIR / "spotlight_baseline_explanations.json"
json_path.write_text(json.dumps(artifacts, indent=2))
print(f"Saved -> {json_path}")

df
